# IMU 병동 전이 인식 — v2 재현 노트북 (JupyterLab)

*Sensor choice over model scale: gyroscope-dependent recognition of ward
bed-exit transitions from a wearable IMU* (MDPI *Sensors*, v2)

이 노트북은 **v2 논문(IMU 단독, ECG 분석 제외)** 의 모든 그림·표·수치를 재현합니다.
①–⑥은 CPU만으로 수 분 안에 돌아가며 §3.1–§3.4와 그림 1–7·표 2를 만듭니다.
⑦–⑨는 파운데이션 모델 단계로 §3.5·표 1·3·그림 8·보충 그림 S1을 완성합니다.

**GPU가 실제로 필요한 단계는 ⑦(임베딩 추출)과 ⑨(완전 미세조정) 둘뿐입니다.**
`results/unimts_emb_w128.npz`가 이미 있으면 ⑧은 CPU로 돌아가며, frozen probe와
embedding 판독이 보고 값을 네 자리까지 재현합니다.

마지막 셀의 **재현 검증**이 산출된 수치를 논문에 실린 값과 자동 대조합니다.

> **실행 순서 주의:** 아래 셀 0(환경)과 헬퍼 셀을 먼저 실행하세요. 모든 단계가 거기서
> 정의하는 `run()` 을 씁니다. 중간부터 돌리려면 메뉴 **Run ▸ Run All Above Selected Cell**.


## 0. 환경 설정 · 데이터 확인

In [ ]:
%automagic off   # run(...) 이 IPython의 %run 매직으로 오해되는 것을 막음

import os, sys, subprocess, glob, json
from pathlib import Path

REPO = Path.cwd()
assert (REPO/'src').exists(), f'src/ 없음 — 노트북을 저장소 루트에서 여세요. 현재: {REPO}'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
sys.path.insert(0, str(REPO/'src'))
print('repo  :', REPO)
print('python:', sys.version.split()[0])

# data/ 가 없으면 옆 폴더(imu_ward_har/data)를 자동 연결(정션)
DATA = REPO/'data'
if not DATA.exists():
    alt = REPO.parent/'imu_ward_har'/'data'
    if alt.exists():
        print('data/ 없음 → 자동 연결 시도:', alt)
        subprocess.run(['cmd','/c','mklink','/J',str(DATA),str(alt)], capture_output=True, text=True)
print('data  :', 'OK' if DATA.exists() else '없음 — README의 데이터 배치 안내 참고')


## (선택) 패키지 설치 — 파운데이션 모델(⑦–⑨)용
UniMTS 원본 코드와 체크포인트는 이미 저장소의 `external_UniMTS/` 에 있습니다.
아래는 GPU에서 ⑦–⑨를 돌리는 데 필요한 파이썬 패키지입니다. 이미 있으면 건너뛰세요.
`torch.cuda.is_available()` 가 `True` 여야 GPU를 씁니다(⑦–⑨ 첫 줄에서 출력).


In [ ]:
# 핵심 파이프라인(①–⑥)용:
# !pip install -r requirements.txt

# 파운데이션 모델(⑦–⑨)용 — CUDA 12.6 휠 기준:
# !pip install torch==2.13.0+cu126 torchvision --index-url https://download.pytorch.org/whl/cu126
# !pip install ftfy==6.3.1 regex git+https://github.com/openai/CLIP.git

import torch
print('torch', torch.__version__, '| CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU  :', torch.cuda.get_device_name(0))


## 실행 헬퍼
각 스크립트를 현재 커널의 파이썬으로 실행하고 출력을 스트리밍합니다. **이 셀을 반드시 먼저 실행하세요.**

In [ ]:
FAILED = []   # 실패한 단계 기록 — 마지막 검증 셀이 모아서 보고합니다

def run(*args):
    cmd = [sys.executable] + list(args)
    print('>>>', ' '.join(args)); sys.stdout.flush()
    r = subprocess.run(cmd, cwd=str(REPO), capture_output=True, text=True,
                       env={**os.environ, 'PYTHONIOENCODING':'utf-8'})
    if r.stdout:
        print(r.stdout[-6000:])
    if r.returncode:
        # 일부러 예외를 올리지 않습니다(뒤 단계가 계속 돌도록). 대신 FAILED 에 쌓아
        # 마지막 검증 셀이 보고합니다 — '예외 0건'을 성공으로 오독하지 않도록.
        FAILED.append((' '.join(args), r.returncode))
        print('!! 종료코드', r.returncode); print('STDERR:'); print(r.stderr[-3000:])
    return r.returncode


## 핵심 파이프라인 (CPU만으로 재현 — ①–⑥)
GPU 없이 §3.1–§3.4와 그림 1–7·표 2가 모두 나옵니다.

### ① HAPT 윈도 생성 (32/64/96/128 샘플 = 0.64–2.56 s)
주 분석은 128(2.56 s). ④ 창 스윕이 네 길이를 모두 요구하므로 한 번에 생성합니다.

In [ ]:
run('src/prepare_hapt.py','--wins','32','64','96','128')

### ①-b 12-class HAPT 베이스라인 — §3.1 / 그림 2
표준 12-class 헤드라인(정확도 0.937, fold macro-F1 0.854)을 클래스별로 분해합니다.
CPU 히스토그램 빌더로 결정적. `--device cuda` 는 빠르지만 값이 미세하게 달라집니다.

In [ ]:
run('src/run_e1.py')   # CPU만 있으면 그대로; GPU 속도가 필요하면 --device cuda

### ② 자이로 절제 — §3.2 / 그림 4 (대표 발견)
6축 vs 3축: macro-F1 0.822→0.647, 침대 이탈 0.657→0.351.

In [ ]:
run('src/run_ablation.py','--wins','128')

### ③ 10-class 과제 + 혼동행렬 — §3.1 / 그림 3
피험자별 macro-F1 0.822, 침대 이탈 0.657; 혼동행렬(그림 3).

In [ ]:
run('src/run_transitions.py','--wins','128')

### ④ 창 길이 스윕 — §3.3 / 그림 6 · 표 2
0.64→2.56 s: macro-F1 0.720→0.833 (생존편향 제거).

In [ ]:
run('src/run_window_fair.py')

### ⑤ 라벨 효율 — §3.4 / 그림 7
`run_fm_compare.py` 가 임베딩 없이 수작업 6축·3축 팔만 돌아 **GPU 없이 §3.4와 그림 7**을 냅니다
(3 시드 평균). 5분→전체: macro-F1 0.435→0.815, 침대 이탈 0.180→0.640, bout 2.7→25.1→91.2.
임베딩이 없으면 전체 라벨 결과는 `fm_compare_handonly_w128.json` 으로 저장되어 §3.5용 4팔 산출물을 덮어쓰지 않습니다.

In [ ]:
run('src/run_fm_compare.py','--win','128')

### ⑥ 그림·표 생성 — 그림 1–8 · S1, 표 1–3
`make_figures`·`make_extra_figures`가 그림 1–4·7–8과 보충 그림 S1, 표 1·3을,
`make_results_extras`가 **그림 5(피험자별 대응)·그림 6·표 2(창 길이)**를 만듭니다.
그림 5는 피험자별 6축/3축 점수를 재계산하므로 2–3분 걸립니다.

표 1은 파운데이션 모델 산출물(⑦–⑧)이 있어야 나옵니다. 그림 8·S1은 그 산출물이 없으면
보관된 `table2_headline.csv`에서 되그리며, 그때는 그렇게 했다고 화면에 알립니다.

마지막 줄이 저장소 `figures/`·`results/` 산출물을 원고 자산 폴더로 동기화합니다.
`manuscript-only`가 0이면 원고의 모든 그림·표를 이 저장소가 만들어냈다는 뜻입니다.


In [ ]:
run('src/make_figures.py')
run('src/make_extra_figures.py')
run('src/make_results_extras.py')

# 저장소 산출물 → 원고 자산 폴더(manuscript/analysis/figures, analysis/tables) 동기화
print(subprocess.run([sys.executable, str(REPO.parent/'sync_manuscript_assets.py')],
                    capture_output=True, text=True).stdout)

## 파운데이션 모델 (⑦–⑨) — GPU
§3.5·표 1·3·그림 8·보충 그림 S1을 완성합니다.
`external_UniMTS/` 의 원본 코드·체크포인트(UniMTS.pth)와 위 셀의 torch/CLIP이 있어야 합니다.

**⑦(임베딩)과 ⑨(완전 미세조정)만 GPU가 필요합니다.** ⑧은 CPU로 돕니다.
각 셀은 실행 전에 보관된 산출물(임베딩·보고 값 JSON)을 스냅샷으로 남겨,
재실행이 보고 값을 지우지 않게 합니다.


### 검증: 공개 UniMTS 체크포인트는 3채널(가속도 전용)
첫 층 배치 정규화 폭 66 = 22관절 × 3 → 자이로 채널이 들어갈 자리가 구조적으로 없음.

In [ ]:
run('src/check_unimts_ckpt.py')

### ⑦ UniMTS 임베딩 추출 — GPU
`external_UniMTS/checkpoint/UniMTS.pth` 로 임베딩을 새로 만듭니다.
실행 전에 보관본을 `unimts_emb_w128.archived.npz` 로 스냅샷하고, 실행 후 새 임베딩이
보관본과 같은지 확인합니다. 같으면(예상되는 경우) ⑧이 재계산 없이 넘어갑니다.


In [ ]:
import shutil, numpy as np
_emb = REPO/'results'/'unimts_emb_w128.npz'
_arch = REPO/'results'/'unimts_emb_w128.archived.npz'
if _emb.exists() and not _arch.exists():
    shutil.copy2(_emb, _arch)
    print('보관 임베딩 스냅샷 ->', _arch.name)

rc = run('src/unimts_embed.py','--win','128','--gyro','0','--padding','64')

EMB_CHANGED = True
if rc == 0 and _arch.exists() and _emb.exists():
    a = np.load(_arch); b = np.load(_emb)
    key = 'emb' if 'emb' in a.files else a.files[0]
    d = float(np.abs(a[key].astype('float64') - b[key].astype('float64')).max())
    EMB_CHANGED = d > 1e-5
    print(f'새 임베딩 vs 보관본 최대 절대차 {d:.2e}  ->',
          '동일 (⑧ 재계산 불필요)' if not EMB_CHANGED else '차이 있음 (⑧ 재계산 필요)')
elif rc != 0:
    print('⑦ 실패 — 보관된 임베딩을 그대로 사용합니다.')
    EMB_CHANGED = False


### ⑧ 표현 비교 (전체 4팔) — §3.5 / 표 1·3 / 그림 8 · S1
⑤에서 이미 4팔이 계산됩니다. **⑦로 임베딩이 실제로 바뀐 경우에만** 다시 계산하고,
아니면 건너뜁니다(라벨 효율까지 재계산하는 50분짜리 중복을 피함).
`FORCE = True` 로 무조건 재계산할 수 있습니다. 이 단계는 GPU 없이 돕니다.


In [ ]:
FORCE = False   # 무조건 재계산하려면 True

_fm = REPO/'results'/'fm_compare_w128.json'
_have4 = False
if _fm.exists():
    _j = json.loads(_fm.read_text())
    _have4 = all(k in _j for k in ('unimts3+logreg', 'unimts3+xgb'))
_emb_changed = globals().get('EMB_CHANGED', False)

if _have4 and not _emb_changed and not FORCE:
    print('건너뜀 — 4팔 결과가 이미 있고 임베딩도 그대로입니다.')
    for k in ('hand6+xgb', 'hand3+xgb', 'unimts3+xgb', 'unimts3+logreg'):
        v = _j[k]
        print(f"    {k:16} macro-F1 {v['macro_f1_subject_mean']:.4f}"
              f"   bed-exit {v['bed_exit_f1_subject_mean']:.4f}")
else:
    why = 'FORCE' if FORCE else ('임베딩 변경' if _emb_changed else '4팔 없음')
    print(f're-run run_fm_compare ({why}) — 라벨 효율까지 다시 계산되어 수십 분 걸립니다.')
    run('src/run_fm_compare.py','--win','128')


### ⑨ 완전 미세조정 — §3.5 (GPU)
5-fold 각 폴드 약 8분. **CUDA 미세조정은 bit-재현이 되지 않습니다**(README 참고):
보고 값(macro 0.7726 / bed 0.5550)과 약 0.002 이내에서 달라질 수 있으며, 이는 오류가
아닙니다. 실행 전에 보고 값 JSON을 `finetune_full_w128.reported.json` 으로 스냅샷하므로
보고 값은 항상 복구 가능합니다.

`RUN_FULL = True` 가 전체 실행, `False` 면 폴드 1만 도는 `--smoke`(파일 미기록)입니다.

> 주의: 이 셀이 `finetune_full_w128.json` 을 새 값으로 덮어씁니다. 원고와 똑같은
> 그림·표를 원하면 ⑥을 다시 돌리기 전에 위 스냅샷을 되돌리세요.


In [ ]:
RUN_FULL = True   # False 면 --smoke (폴드 1만, 파일 미기록)

import shutil
_ft = REPO/'results'/'finetune_full_w128.json'
_ftrep = REPO/'results'/'finetune_full_w128.reported.json'
if _ft.exists() and not _ftrep.exists():
    shutil.copy2(_ft, _ftrep)
    print('보고 값 스냅샷 ->', _ftrep.name)

if RUN_FULL:
    run('src/run_finetune.py','--mode','full','--epochs','20','--padding','64')
else:
    run('src/run_finetune.py','--mode','full','--smoke')   # 결과 파일 미기록


## 결과 그림 미리보기 (v2 — 본문 8개 + 보충 S1)

In [ ]:
from IPython.display import Image, display
FIGDIR = REPO/'figures'
v2_figs = ['fig_pipeline.png','fig1_headline_decomposition.png','fig_confusion.png',
           'fig2_gyro_ablation.png','fig_gyro_paired.png','fig_window_sweep.png',
           'fig3_label_efficiency.png','fig4_representation.png']
for i,f in enumerate(v2_figs,1):
    p = FIGDIR/f
    print(f'그림 {i}: {f}', '' if p.exists() else '  (아직 생성 안 됨)')
    if p.exists(): display(Image(filename=str(p), width=560))

pS1 = FIGDIR/'figS1_representation_scatter.png'
print('보충 그림 S1: figS1_representation_scatter.png',
      '' if pS1.exists() else '  (아직 생성 안 됨)')
if pS1.exists(): display(Image(filename=str(pS1), width=560))


## 재현 검증 — 산출 수치 대 논문 기재 값

아래 셀은 방금 만들어진 결과 파일을 읽어 논문에 실린 값과 대조합니다.
`X`가 하나라도 뜨면 그 줄의 산출물이 보고본과 다르다는 뜻이므로, 원고를 고치기 전에
어느 실행이 보고본인지부터 확인하세요 (`results/*_log.txt`의 기록이 근거가 됩니다).


In [ ]:
import json
import pandas as pd

RES = REPO/'results'

# 논문에 실린 값 (macro-F1, bed-exit F1)
REPORTED = {
    'hand6+xgb':       (0.8225, 0.6575),
    'hand3+xgb':       (0.6467, 0.3512),
    'unimts3+xgb':     (0.7010, 0.4432),
    'unimts3+logreg':  (0.7310, 0.5345),
    'unimts3+ft-full': (0.7726, 0.5550),
}

rows = []
fm = RES/'fm_compare_w128.json'
if not fm.exists():
    fm = RES/'fm_compare_handonly_w128.json'
if fm.exists():
    j = json.loads(fm.read_text())
    for k, v in j.items():
        if isinstance(v, dict) and 'macro_f1_subject_mean' in v:
            rows.append((k, v['macro_f1_subject_mean'], v['bed_exit_f1_subject_mean']))
ft = RES/'finetune_full_w128.json'
if ft.exists():
    j = json.loads(ft.read_text())
    rows.append(('unimts3+ft-full', j['macro_f1_mean'], j['bed_exit_f1_mean']))

print(f'{"arm":18} {"산출 macro/bed":>16}  {"보고 macro/bed":>16}  판정')
bad = 0
for k, m, b in rows:
    rm, rb = REPORTED[k]
    # CUDA fine-tuning is not bit-reproducible (README): allow ~0.002 for that row only
    tol = 3e-3 if k == 'unimts3+ft-full' else 5e-4
    ok = abs(m - rm) < tol and abs(b - rb) < tol
    bad += (not ok)
    note = '  (CUDA 재현오차 허용)' if k == 'unimts3+ft-full' else ''
    print(f'{k:18} {m:.4f}/{b:.4f}      {rm:.4f}/{rb:.4f}      {"O" if ok else "X"}{note}')

missing = [k for k in REPORTED if k not in {r[0] for r in rows}]
print()
if missing:
    print('아직 산출 안 됨(⑦–⑨ 미실행):', ', '.join(missing))
print('불일치' , bad, '건' if bad else '건 — 모두 보고 값과 일치')

print()
print('=== 단계 실행 상태 ===')
if not FAILED:
    print('모든 단계가 종료코드 0 으로 끝났습니다.')
else:
    print(f'종료코드가 0 이 아닌 단계 {len(FAILED)}건 '
          '— 위 표가 "일치"여도 그 단계는 실제로 돌지 않았습니다:')
    for cmd, rc in FAILED:
        print(f'  [{rc}] {cmd}')
    print()
    print('CPU 전용 PC에서는 torch/UniMTS 원본 코드가 필요한 ⑦·⑨와 체크포인트 검증이')
    print('실패하는 것이 정상입니다. 그때 표의 unimts3+ft-full 행은 이번에 계산된 값이')
    print('아니라 보관된 finetune_full_w128.json 을 읽은 것입니다.')
